# Purpose

The purpose of this notebook is to teach an agent to trade on NVDA and PLTR using the ema 9 and 20 stategy with a combination of vwap and other indicators
## To get started, Alpaca

Alpaca is what we are using to get our data, we will be using the "SIP" data feed as it is the one where we can get the most data for free and from previous experience, we can get the data for up to 10 years in the past, for our purposes it will be enough.
### The theory:
We can tech a RL agent to trade using strategy like the ema 9, 20 and other signals for it to perform profitably.
 

### this cell holds most the libraries 

In [1]:
#general libraries used 
import pandas as pd
import numpy as np
# alpaca libraries used
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from datetime import datetime
#os
import os

### This cell holds our account information and the data feed (global variables)

In [2]:
## URL of alpaca
BASE_URL = "https://paper-api.alpaca.markets"
#alpaca secret key and key 
ALPACA_API_KEY = "YOUR ALPACA KEY"
ALPACA_SECRET_KEY = "YOUR ALPACA SECRET KEY"
#data feed chosen
DATA_FEED = 'sip'
#to find your file location, you can go to yoru terminal, navigate to where you want the file and type "pwd", copy the path and
#change base_folder to that path
#base raw data folder path, where we will store our raw data
BASE_FOLDER_RAW = "/Users/rchbeir/Desktop/trading-bt/notebooks/Learning/Data/raw data"
#base processed data folder path, where we will store our processed data
BASE_FOLDER_PROCESSED = "/Users/rchbeir/Desktop/trading-bt/notebooks/Learning/Data/processed data"
#the symbols we are working with:
SYMBOLS = ["NVDA", "PLTR"]
#initializing an empty dictionnary so our data is more organized
all_data = {} 
#initializing an empty dictionnary so our feature data is more organized
all_features = {}
#the amount of days we are pulling data from
START = "2016-01-01"
END = "2026-01-30"

### Next cell defines the variables we want
We want to look at NVDA and PLTR for the last 10 years, but in case we want to get more tickers in the future, we need to define a function that gets the data for any ticker we want so we could just call on that function

In [3]:
def get_data_5m(symbol: str, start: str,end: str):
    client = StockHistoricalDataClient(ALPACA_API_KEY, ALPACA_SECRET_KEY)
    print("Connected to client")
    request_params = StockBarsRequest(
        symbol_or_symbols=[symbol],
        timeframe = TimeFrame(5, TimeFrameUnit.Minute),
        start=pd.Timestamp(start, tz="UTC"),   # Start date (1 year ago from today)
        end=pd.Timestamp(end, tz="UTC"),     # End date (today)
        feed = DATA_FEED,
        )
    print("Loaded request params")
    print("fetching the 5 min data for",symbol)
    print("this might take a while ...")
    bars = client.get_stock_bars(request_params)
    print(symbol,"5 min data fetched")
    bars = bars.df
    print(bars.head())
    return bars
    
    

Testing our function

In [4]:
# getting the data between jan 1 2025 and jan 1 2026
start = "2025-12-01"
end = "2026-01-01"
symbol = "NVDA"
#using the function above
NVDA_df = get_data_5m(symbol, start, end)
#checking out the features
NVDA_df.columns

Connected to client
Loaded request params
fetching the 5 min data for NVDA
this might take a while ...
NVDA 5 min data fetched
                                    open    high     low   close   volume  \
symbol timestamp                                                            
NVDA   2025-12-01 09:00:00+00:00  175.65  176.40  175.05  175.15  82827.0   
       2025-12-01 09:05:00+00:00  175.15  175.60  175.15  175.51  41143.0   
       2025-12-01 09:10:00+00:00  175.55  175.59  175.22  175.32  25040.0   
       2025-12-01 09:15:00+00:00  175.33  175.41  175.17  175.17  24855.0   
       2025-12-01 09:20:00+00:00  175.15  175.38  175.11  175.37  18893.0   

                                  trade_count        vwap  
symbol timestamp                                           
NVDA   2025-12-01 09:00:00+00:00       2377.0  175.341606  
       2025-12-01 09:05:00+00:00        982.0  175.415022  
       2025-12-01 09:10:00+00:00        712.0  175.361914  
       2025-12-01 09:15:00+00:00 

Index(['open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap'], dtype='object')

### Great, we see that it worked and that we got the data, now we want a function that could store the df passed to it in our data file so we don't have to redownload the data multiple times

In [4]:
def store_data(df,symbol, base_folder):
    done = False
    #creating the full path
    full_path = os.path.join(base_folder, f"{symbol}.csv")
    answer = ''
    if (os.path.exists(full_path)):
        print("there is already a file there with the same name, are you sure you want to override it?")
        while (done == False):
            answer = input('type yes to override the current file and no to stop upload:').lower()
            if (answer == "yes"):
                print("Uploading...")
                df.to_csv(full_path)
                print("Uploading done")
                print("path:",full_path)
                done = True
            if (answer == "no") :
                print("Upload cancelled")
                done = True
    else:
        print("Uploading...")
        df.to_csv(full_path)
        print("Uploading done")
        print("path:",full_path)

In [45]:
#changing our folder path
base_path = BASE_FOLDER_RAW
store_data(NVDA_df,symbol,base_path)

there is already a file there with the same name, are you sure you want to override it?


type yes to override the current file and no to stop upload: no


Upload cancelled


### Now that we have our cell that can upload the raw data to our base_folder, we need to Download all the data for NVDA and PLTR that we will be using to train our model
The cell below does exactly that!

In [9]:
start = START
end = END
symbols = SYMBOLS
base_path = BASE_FOLDER_RAW
for s in symbols:
    df = get_data_5m(s, start, end)
    store_data(df,s,base_path)

Connected to client
Loaded request params
fetching the 5 min data for NVDA
this might take a while ...
NVDA 5 min data fetched
                                   open   high    low  close  volume  \
symbol timestamp                                                       
NVDA   2016-01-04 11:35:00+00:00  32.40  32.40  32.40  32.40   200.0   
       2016-01-04 12:10:00+00:00  32.30  32.30  32.30  32.30   145.0   
       2016-01-04 12:35:00+00:00  32.30  32.30  31.95  31.95  1695.0   
       2016-01-04 12:40:00+00:00  31.94  31.95  31.94  31.95   500.0   
       2016-01-04 13:00:00+00:00  32.59  32.59  32.20  32.20   450.0   

                                  trade_count       vwap  
symbol timestamp                                          
NVDA   2016-01-04 11:35:00+00:00          1.0  32.400000  
       2016-01-04 12:10:00+00:00          1.0  32.300000  
       2016-01-04 12:35:00+00:00         13.0  32.061046  
       2016-01-04 12:40:00+00:00          2.0  31.946000  
       2016-01

type yes to override the current file and no to stop upload: yes


Uploading...
Uploading done
path: /Users/rchbeir/Desktop/trading-bt/notebooks/Learning/Data/raw data/NVDA.csv
Connected to client
Loaded request params
fetching the 5 min data for PLTR
this might take a while ...
PLTR 5 min data fetched
                                     open   high    low   close      volume  \
symbol timestamp                                                              
PLTR   2020-09-30 17:35:00+00:00  10.0000  11.00  10.00  10.300  70983246.0   
       2020-09-30 17:40:00+00:00  10.2675  11.30  10.04  11.230  42428035.0   
       2020-09-30 17:45:00+00:00  11.2010  11.37  10.83  11.200  20596270.0   
       2020-09-30 17:50:00+00:00  11.2050  11.42  10.75  10.890  17372931.0   
       2020-09-30 17:55:00+00:00  10.8800  11.00  10.60  10.605  10729639.0   

                                  trade_count       vwap  
symbol timestamp                                          
PLTR   2020-09-30 17:35:00+00:00      26006.0  10.066837  
       2020-09-30 17:40:00+00:00

type yes to override the current file and no to stop upload: yes


Uploading...
Uploading done
path: /Users/rchbeir/Desktop/trading-bt/notebooks/Learning/Data/raw data/PLTR.csv


### Great, now we have our raw data and we know exactly where it is, now it is time to clean up our data, for that we must remove all the days that have NaN values
but first, we want to store the csv data in our dict so we are able to access whatever is in our csv file

In [5]:
symbols = SYMBOLS
for s in symbols:
    # setting the main path
    main_path = BASE_FOLDER_RAW
    #joining it with the symbols we took out 
    full_path = os.path.join(main_path, f"{s}.csv")
    print(f"Getting data from {full_path}")
    df = pd.read_csv(full_path)
    print(f"{s} data fetched")
    # Store it in the dictionary
    all_data[s] = df 

#making sure we can have the data in our all_data dict the data easily
print(list(all_data.keys()))
# now we can use it like this
#print(all_data['NVDA'].head())
# all_data['NVDA'] is equivalent to a dict

Getting data from /Users/rchbeir/Desktop/trading-bt/notebooks/Learning/Data/raw data/NVDA.csv
NVDA data fetched
Getting data from /Users/rchbeir/Desktop/trading-bt/notebooks/Learning/Data/raw data/PLTR.csv
PLTR data fetched
['NVDA', 'PLTR']


### Now that we got the data, we need to process it, but first, we need to check the percentage of days that have missing parameters


In [35]:
#printing all the NANs if any
print(all_data['NVDA'].isna().sum())
print(all_data['PLTR'].isna().sum())

symbol         0
timestamp      0
open           0
high           0
low            0
close          0
volume         0
trade_count    0
vwap           0
dtype: int64
symbol         0
timestamp      0
open           0
high           0
low            0
close          0
volume         0
trade_count    0
vwap           0
dtype: int64


As we can see, there is no NANs but that doesn't garantee that we don't have missing days, the cell below checks for missing days

In [36]:
import pandas as pd
import datetime

def remove_missing_days(df, s):
    if df is None or len(df) == 0:
        return pd.DataFrame()

    print(f"Processing {s}...")

    # 1. Convert to Datetime and Ensure UTC
    # If your data is already timezone-aware, this just standardizes it. 
    # If it's naive (no timezone), we tell pandas "This is UTC".
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    if df['timestamp'].dt.tz is None:
        df['timestamp'] = df['timestamp'].dt.tz_localize('UTC')
    else:
        df['timestamp'] = df['timestamp'].dt.tz_convert('UTC')

    # 2. Create a "Local Time" Column (PST/PDT)
    # This handles Daylight Savings automatically. 
    # 4:45 AM will always be 4:45 AM in this column, regardless of the date.
    df['local_time'] = df['timestamp'].dt.tz_convert('America/Los_Angeles')

    # 3. Filter for the Specific Time Window (04:45 to 13:00)
    # We define the specific time objects
    start_time = datetime.time(4, 45)
    end_time = datetime.time(13, 0) # 1:00 PM

    # Create the mask: Keep rows >= 4:45 AND <= 1:00 PM
    print("Filtering rows to 04:45 - 13:00 PST/PDT...")
    time_mask = (df['local_time'].dt.time >= start_time) & \
                (df['local_time'].dt.time <= end_time)
    
    # Apply the time filter immediately
    df_session = df[time_mask].copy()

    # 4. Group by Date to Count Bars
    df_session['date'] = df_session['local_time'].dt.date
    daily_counts = df_session.groupby('date').size()

    # 5. Detect Expected Count
    # 4:45 to 13:00 is 495 minutes. Divided by 5 = 99 intervals.
    # However, depending on if your data includes the 13:00 print, it might be 100.
    # We let .mode() find the truth.
    if len(daily_counts) == 0:
        print(f"Warning: No data found in the 04:45-13:00 window for {s}")
        return pd.DataFrame()

    expected_count = daily_counts.mode()[0]
    print(f"Expected 5-min bars for full session: {expected_count}")

    # 6. Filter for Complete Days Only
    # We only keep dates that have exactly the expected count, (it discards the right cols)
    valid_dates = daily_counts[daily_counts == expected_count].index
    
    print(f"Found {len(valid_dates)} complete days out of {len(daily_counts)} total days")

    # 7. Create Final Clean DataFrame
    df_clean = df_session[df_session['date'].isin(valid_dates)].copy()

    # Cleanup helper columns
    df_clean.drop(columns=['date', 'local_time'], inplace=True)
    
    # (Optional) Convert back to UTC if you prefer your final data in UTC
    # df_clean['timestamp'] is still UTC, so we are good.

    return df_clean

Now we use the function above to clean up the data:

In [37]:
for s in SYMBOLS:
    #checking if s is in all data
    if s in all_data:
        all_data[s] = remove_missing_days(all_data[s], s)
    else:
        print(f"Warning: No data found for {s}")


Processing NVDA...
Filtering rows to 04:45 - 13:00 PST/PDT...
Expected 5-min bars for full session: 100
Found 1628 complete days out of 2533 total days
Processing PLTR...
Filtering rows to 04:45 - 13:00 PST/PDT...
Expected 5-min bars for full session: 100
Found 1204 complete days out of 1339 total days


In [38]:
print(all_data['NVDA'].head())
print(all_data['NVDA'].tail())

     symbol                 timestamp   open   high    low  close   volume  \
2889   NVDA 2016-02-18 12:45:00+00:00  29.82  29.82  29.68  29.73   5504.0   
2890   NVDA 2016-02-18 12:50:00+00:00  29.73  29.74  29.62  29.74   4408.0   
2891   NVDA 2016-02-18 12:55:00+00:00  29.74  29.74  29.74  29.74   1035.0   
2892   NVDA 2016-02-18 13:00:00+00:00  29.70  30.03  29.66  29.88  39864.0   
2893   NVDA 2016-02-18 13:05:00+00:00  29.86  29.90  29.80  29.80   5635.0   

      trade_count       vwap  
2889         17.0  29.755625  
2890         28.0  29.720000  
2891          3.0  29.740000  
2892         91.0  29.792445  
2893         26.0  29.844374  
       symbol                 timestamp    open    high      low    close  \
404608   NVDA 2026-01-29 20:40:00+00:00  191.45  191.60  190.925  190.995   
404609   NVDA 2026-01-29 20:45:00+00:00  191.00  191.20  190.880  191.125   
404610   NVDA 2026-01-29 20:50:00+00:00  191.13  192.00  191.130  191.850   
404611   NVDA 2026-01-29 20:55:00+00:

Now, we can save our processed data for use 

In [39]:
base_path = BASE_FOLDER_PROCESSED
for s in SYMBOLS:
    store_data(all_data[s],s, base_path)

there is already a file there with the same name, are you sure you want to override it?


type yes to override the current file and no to stop upload: yes


Uploading...
Uploading done
path: /Users/rchbeir/Desktop/trading-bt/notebooks/Learning/Data/processed data/NVDA.csv
there is already a file there with the same name, are you sure you want to override it?


type yes to override the current file and no to stop upload: yes


Uploading...
Uploading done
path: /Users/rchbeir/Desktop/trading-bt/notebooks/Learning/Data/processed data/PLTR.csv


### Now we gotta load the right data so we are working on loaded data and not temp data. run the cell below

In [6]:
symbols = SYMBOLS
for s in symbols:
    # setting the main path
    main_path = BASE_FOLDER_PROCESSED
    #joining it with the symbols we took out 
    full_path = os.path.join(main_path, f"{s}.csv")
    print(f"Getting data from {full_path}")
    df = pd.read_csv(full_path)
    print(f"{s} data fetched")
    # Store it in the dictionary
    all_data[f"Processed_{s}"] = df 

#making sure we can have the data in our all_data dict the data easily
print(list(all_data.keys()))

Getting data from /Users/rchbeir/Desktop/trading-bt/notebooks/Learning/Data/processed data/NVDA.csv
NVDA data fetched
Getting data from /Users/rchbeir/Desktop/trading-bt/notebooks/Learning/Data/processed data/PLTR.csv
PLTR data fetched
['NVDA', 'PLTR', 'Processed_NVDA', 'Processed_PLTR']


### Great! we got all the data we need processed and ready, now we want to calculate the indicators and features we will be feeding into our RL bot. we will go one by one and explain their importance and why we are using them. The cell below will go over them 1 by one and calculate them and store them in a df which we will store for each we are following this spec

# I) STATE (INPUT) SPEC

## 1. Market State (Asset-Level, Normalized & Causal)

### 1.1 Returns (Vol-Normalized, Multi-Horizon Momentum)
All returns are normalized by **intraday volatility** (not overnight gap volatility):

- `r_1n  = r_1  / (vol_intraday_20 + eps)`
- `r_5n  = r_5  / (vol_intraday_20 + eps)`
- `r_20n = r_20 / (vol_intraday_20 + eps)`
- `r_60n = r_60 / (vol_intraday_60 + eps)`
- `r_90n = r_90 / (vol_intraday_90 + eps)`

Purpose: stationarity across regimes (0.05% in low vol ≠ 0.05% in high vol).

---

### 1.2 Volatility (Split: Intraday vs Regime)
- `vol_intraday_20`, `vol_intraday_60`, `vol_intraday_90`  (RTH-only bars)
- `EWMA` (exponential weighted moving average, will be used to get the regime)
- `gap_pct_norm` (overnight gap as explicit feature; NOT in denominators)(currently not vailable with our data)

**Morning Gap Normalization Guardrail**
- Do NOT allow overnight gap to inflate `vol_intraday_*`.
- Compute intraday vols using regular-session bars only.
- Represent the gap separately via `gap_pct_norm`.

---

### 1.3 Candle Microstructure (Impulse vs Rejection)
- `hl_range`
- `candle_body`
- `impulse_ratio`
- `wick_ratio` = (High - max(Open,Close)) / (High - Low + eps)  (rejection / topping)
- `tick_imbalance_proxy` = ((Close - Low) / (High - Low + eps)) * volume_zscore  (pressure proxy)
We can't just give the hl_range and 'candle body like that, we need to normalize them to the volatility.
We need to do this because we can't give it the price of NVDA like 200 today and 10 years ago we were training at 20
---

### 1.4 Volume & Liquidity (Cold-Start Safe)
- `log_volume`
- `volume_zscore`
- `dollar_volume_z`
- `relative_vol_z_timeofday` = (Volume_now - AvgVolume_at_this_time) / (Std_at_this_time + eps)

**Z-Score Cold-Start Guardrails**
- Rolling intraday window (≈ full session) with optional seed from prior session (e.g., last 30 min)
- Enforce floors: `std >= std_floor`, `vol >= vol_floor`
- Require `N_min` bars before trusting intraday-only statistics
- Never normalize using only the current bar at the open

---

## 2. Strategy Signal State (Pruned, Non-Redundant)

### 2.1 EMA Structure (EMA-9 / EMA-20)
- `ema_spread_9_20`
- `ema_spread_slope_9_20`
- `price_to_ema9`
- `ema_stack_alignment`
- `ema_cross_flag_9_20`

### 2.2 VWAP Context
- `vwap_z`
- `vwap_abs_z`
- `vwap_slope`
- `vwap_between_emas_flag`

### 2.3 Trend Exhaustion / Decay
- `acceleration_proxy_norm` = (r_1 - r_5) / (vol_intraday_20 + eps)
- `momentum_decay_corr_20` = corr(price, time) over last 20 bars (trend integrity proxy)

---

## 3. Market Context State (QQQ/SPY/NDX Proxy; Causal)

### 3.1 Market Returns & Vol (Normalized)
- `mkt_r_5n`, `mkt_r_20n` (QQQ returns normalized by QQQ vol_intraday_20)
- `mkt_vol_intraday_20`
- `mkt_vol_regime`

### 3.2 Relationship Features (Asset vs Market)
- `corr_asset_mkt_20` (rolling correlation, asset vs QQQ)
- `beta_proxy` = cov(asset_ret, mkt_ret) / (var(mkt_ret) + eps)
- `rel_strength_60` = (asset_r_60 - mkt_r_60) / (vol_intraday_60 + eps)

### 3.3 Market Regime Flags
- `mkt_above_ema20_flag` (QQQ above/below its EMA-20)
- `mkt_trend_strength` (QQQ trend proxy)

Purpose: avoid “blind agent” errors where single-name setups are invalid during market dumps.

---

## 4. Position State (Per-Trade Memory)

- `position ∈ {−1, 0, +1}`
- `entry_price_distance`
- `unrealized_pnl_norm`
- `time_in_trade`
- `bars_since_last_trade`
- `bars_since_exit`
- `volatility_since_last_trade`
- `trades_taken_today`
- `remaining_trade_budget`
- `time_since_session_high`
- `time_since_session_low`
- `range_position_norm` = (close - session_low) / (session_high - session_low + eps)

`volatility_since_last_trade = clip(mean(vol_intraday_1 over bars since exit), 0, vol_cap)`

---

## 5. Trader Self-Awareness (Stationary, Percent-Based)

All expressed as **percent of equity**.

- `pnl_unrealized_norm`
- `pnl_realized_today_norm`
- `daily_drawdown`
- `risk_budget_left_norm`
- `exposure_fraction`
- `cash_fraction`
- `consecutive_losses` (recent loss streak counter; bounded)

---

## 6. Trade-Level Risk & Opportunity

- `max_adverse_excursion` (vol-normalized)
- `max_favorable_excursion` (vol-normalized)
- `hwm_pnl_real`
- `hwm_pnl_smooth`
- `current_stop_distance`

**Hybrid HWM Logic**
- `hwm_pnl_real`: tracked on real price (close/low) for immediate risk awareness  
- `hwm_pnl_smooth`: tracked on smoothed close (EMA/median) for trend quality  
- Giveback penalties reference `hwm_pnl_smooth`  
- Panic exits reference `hwm_pnl_real`

---

## 7. Time & Opportunity Feasibility (Causal)

- `expected_move_remaining_pct`
- `progress_to_1pct`
- `feasible_1pct_flag`

Feasible if: `k * ATR * sqrt(bars_remaining) >= distance_to_1pct`

---

## 8. Time & Session Context
- `sin(minute_of_day)`
- `cos(minute_of_day)`
- `time_to_close`
- `session_flag`

---

## 9. Regime State
- `trend_strength`
- `range_compression`
- `regime_velocity` (grind-up vs crash-down asymmetry)

---

## 10. Execution & Cost Proxies (Training-Safe)

- `spread_proxy` (do NOT assume equity spread is realistic for options execution)
- `slippage_proxy`

**Training-to-Options Cost Gap Guardrail (Synthetic Friction)**
- Since equity spreads are tiny vs option spreads, inject an additional **per-trade friction cost** in reward on entry/exit:
  - `option_friction_cost_equiv` (constant or mild function of regime)
- Practical target: agent must overcome an effective ~0.3%–0.5% cost per trade in reward terms to discourage scalping.

---

## 11. Action Space
- `EnterLong`
- `EnterShort`
- `Hold`
- `Exit`

---

## 12. Action Masking (Entry-Only)

### 12.1 Budget & Risk Mask
Mask entries if:
- `risk_budget_left_norm <= 0`

### 12.2 Tradeability Mask (Equity Only)
`tradeable_flag = expected_move_remaining_pct >= (spread_proxy + slippage_proxy + edge_buffer)`

If false → mask entries.

### 12.3 Daily Trade Budget with “Mulligan” Probing
Target ~2 trades/day, but allow probing without burning the day:
- If trade duration < 5 minutes AND loss < 0.2% (noise stop), count as 0 or 0.5 trades (implementation choice)
- Otherwise, count normally
Purpose: avoid being locked out after two early shakeouts before the real trend begins.

---

## 13. Policy Architecture (Asymmetry + Long Priority)

- Shared feature encoder  
- Separate actor heads:
  - Long head → `EnterLong`
  - Short head → `EnterShort`
- Shared critic

**Short-Head Learning Weight (Pre-Flight)**
- Scale learning from the Short head by `g_short = 0.5`
- Effective objective: `R_long + g_short * R_short`
- Purpose: prioritize Long performance while retaining bearish structure learning


In [27]:

##################################################################
####                         Helpers                          ####
##################################################################
def roll_std(s, w):
        return s.rolling(window=w, min_periods=w).std()

def roll_sum(s, w):
        return s.rolling(window=w, min_periods=w).sum()




##################################################################
####                                                          ####
####                        1.1, 1.2                          ####
####               Volume Normalized returns                  ####
####                   +Volatility intraday                   ####
####                        + EWMA                            ####
####                                                          ####
##################################################################

def Vol_normalized(df):
    # the volume normalized return is very useful because it takes into account the context of what is going on 
    # in the market. i.e the market is very stable barely any movement and then a sudden rise by 0.5%, it is a breakout
    # but if the market is very noisy and very volatile and a move of 0.5% doesn't count as a move of 0.5% in the last scenario
    # so we feed it the 1, 5, 20, 60 and 90 of this indicator to give it broader market context
    
    # we start by making a copy of the main df so we don't mess with original data
    temp = df.copy()
    # sorting by timestamp to make sure everything good 
    temp = temp.sort_values("timestamp").reset_index(drop=True)

    # A tiny number to prevent "Division by Zero" errors
    epsilon = 1e-6

    # we need to split the data into days before using the rolling window on each
    #making sure everything is in UTC 
    temp['timestamp'] = pd.to_datetime(temp['timestamp'])
    if temp['timestamp'].dt.tz is None:
        temp['timestamp'] = temp['timestamp'].dt.tz_localize('UTC')
    else:
        temp['timestamp'] = temp['timestamp'].dt.tz_convert('UTC')

    #convert to local time
    temp['local_time'] = temp['timestamp'].dt.tz_convert('America/Los_Angeles')

    # making a new col as a datetime
    temp['date'] = temp['local_time'].dt.date

    # getting the log return for each bar, this will leave the first of each day empty
    # here .shift gets the row above by 1 geting the return
    temp['log_ret'] = temp.groupby('date')['close'].transform(lambda s: np.log(s / s.shift(1))).fillna(0.0)

    # the alpha for the EWMA
    alpha = 0.02 

    EWMA = temp.groupby('date')['log_ret'].transform(lambda s: s.shift(1).ewm(alpha=0.02, adjust=False).std())
    
    # Volatility per day only
    # intraday volatility is essentially the standard deviation because we are getting basically how spread apart ae the are
    # the last nth returns around their average. the return here being the close of the preious bar / the close of this bar.
    vol_20 = temp.groupby('date')['log_ret'].transform(lambda s: roll_std(s.shift(1), 20))
    vol_60 = temp.groupby('date')['log_ret'].transform(lambda s: roll_std(s.shift(1), 60))
    vol_90 = temp.groupby('date')['log_ret'].transform(lambda s: roll_std(s.shift(1), 90))

    # Sums per day only 
    sum_5  = temp.groupby('date')['log_ret'].transform(lambda s: roll_sum(s.shift(1), 5))
    sum_20 = temp.groupby('date')['log_ret'].transform(lambda s: roll_sum(s.shift(1), 20))
    sum_60 = temp.groupby('date')['log_ret'].transform(lambda s: roll_sum(s.shift(1), 60))
    sum_90 = temp.groupby('date')['log_ret'].transform(lambda s: roll_sum(s.shift(1), 90))
    
    # Create a new DataFrame to hold just our shiny new features, it includes only our first column of data which is our timestamps 
    features = pd.DataFrame({"timestamp": temp["timestamp"]})
    features ['EWMA'] = EWMA
    features ['vol_20'] = vol_20
    features ['vol_60'] = vol_60
    features ['vol_90'] = vol_90
    features['r_1n']  = temp['log_ret'] / (vol_20 + epsilon)
    features['r_5n']  = sum_5  / (vol_20 + epsilon)
    features['r_20n'] = sum_20 / (vol_20 + epsilon)

    features['r_60n'] = sum_60 / (vol_60 + epsilon)
    features['r_90n'] = sum_90 / (vol_90 + epsilon)


    # use below for debugging
    #print(f"checking out the data nd see if it behaved as we want")
    #print(features.head(21))
    
    # Here we can change the nans to 0 as it would be considered the average of the returns and also
    # when the market opens (our best window) the r_vol 90 and 60 don't really matter that much because we are looking at the micro and
    # not the macro
    # Fill the empty startup periods with 0 (Neutral)
    features.fillna(0.0, inplace=True)
    # Use below for debugging
    #print("checking again to see if the NA has been filled with 0s")
    #print (features.head(21))
    return features

##################################################################
####                                                          ####
####                        1.3 + 1.4                         ####
####                  Candle Microstructure                   ####
####                         (1.4 ip                          ####
##################################################################

def Candle_micro(df):
    temp = df.copy()
    #sorting by timestamp to make sure we have everything good 
    temp = temp.sort_values("timestamp").reset_index(drop=True)

    # Same block as before
    # A tiny number to prevent "Division by Zero" errors
    epsilon = 1e-6

    # we need to split the data into days before using the rolling window on each
    #making sure everything is in UTC 
    temp['timestamp'] = pd.to_datetime(temp['timestamp'])
    if temp['timestamp'].dt.tz is None:
        temp['timestamp'] = temp['timestamp'].dt.tz_localize('UTC')
    else:
        temp['timestamp'] = temp['timestamp'].dt.tz_convert('UTC')  
    #convert to local time
    temp['local_time'] = temp['timestamp'].dt.tz_convert('America/Los_Angeles')

    # making a new col as a datetime
    temp['date'] = temp['local_time'].dt.date
    #getting the log return
    temp['log_ret'] = temp.groupby('date')['close'].transform(lambda s: np.log(s / s.shift(1))).fillna(0.0)
    # using vol20 to normalize
    vol_20 = temp.groupby('date')['log_ret'].transform(lambda s: roll_std(s.shift(1), 20))
    # creating the features column to make it start with the timestamp as a column so we can load days and not just count shit 
    features = pd.DataFrame({"timestamp": temp["timestamp"]})
    grouped = temp.groupby('date')
    ## Calculating the things normalized
    expected_move = temp['close'] * vol_20

    # Here we don't need groupby because we are only using each bar's information.
    features['hl_range'] =(temp['high'] - temp['low']) / (expected_move + epsilon)
    features['candle_body'] = (abs(temp['close'] - temp['open'])) / (expected_move + epsilon)
    features ['impulse_ratio'] = (temp['close'] - temp['open']) / ( temp['high'] - temp['low'] + epsilon)
    features ['up_wick_ratio'] = (temp['high'] - np.maximum(temp['close'],  temp['open'])) / ( temp['high'] - temp['low'] + epsilon)
    features ['low_wick_ratio'] = (np.minimum(temp['close'],  temp['open']) - temp['low']) / ( temp['high'] - temp['low'] + epsilon)
    features['wick_skew'] = features['up_wick_ratio'] - features['low_wick_ratio']


    # Fill the empty startup periods with 0 (Neutral)
    features.fillna(0.0, inplace=True)

    return features


##################################################################
####                                                          ####
####               The function to calculate all              ####
####                      feature cols                        ####
####                                                          ####
##################################################################
    
def make_features(df, s):
    # first feature on our hands is the Volume normalized returns, you can find more details on each
    # type of return in their respective helper functions
    return

In [28]:
f = Vol_normalized(all_data["Processed_NVDA"])
v =  Candle_micro(all_data["Processed_NVDA"])
merged = f.merge(v, on="timestamp", how="inner")
store_data(merged,'test',"/Users/rchbeir/Desktop/trading-bt/notebooks/Learning/Data/processed data")

there is already a file there with the same name, are you sure you want to override it?


type yes to override the current file and no to stop upload: yes


Uploading...
Uploading done
path: /Users/rchbeir/Desktop/trading-bt/notebooks/Learning/Data/processed data/test.csv
